# chat_client.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**

這是終端機版聊天室,對照 DAY5 的 ChatClient.py。原本一定要先開 ChatServer,客戶端還要自己開執行緒收訊息;這裡沒有伺服器,paho-mqtt 套件會幫你開背景執行緒。只要房間一樣,它就能和網頁版 chat.html 互相聊天,因為 MQTT 只看主題,不管對方是網頁還是 Python。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/1)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'chat_client.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 說明與匯入
開頭註解寫了安裝和執行方式。除了 json、sys、time,還用到 uuid 產生不會重複的編號。最重要的是 paho-mqtt,它是 Python 最常用的 MQTT 套件,要先用 pip 安裝;import 時取綽號叫 mqtt,後面用起來比較短。

In [ ]:
# MQTT 聊天室(終端機版)— 跟 chat.html 在同一個房間就能互相聊天
# 對照 DAY5 的 ChatServer.py + ChatClient.py:這裡「沒有伺服器程式」,廣播交給 broker
# 安裝:pip install paho-mqtt
# 執行:python chat_client.py 你的暱稱 [房間名稱]

import json
import sys
import time
import uuid

### 設定:跟網頁同一套主題
暱稱和房間從命令列參數讀,沒給就問或用 demo。主題前綴的格式和 chat.html 一模一樣,兩邊才會碰在一起。注意埠號:Python 不受瀏覽器限制,可以直接用一般 TCP 的 1883;網頁只能走 WebSocket,所以用 8084。MY_ID 是這條連線在 broker 上的身分證,不能和別人重複。

In [ ]:
import paho.mqtt.client as mqtt

# ====== 1. 設定 ======
NAME   = sys.argv[1] if len(sys.argv) > 1 else input("你的暱稱?") or "Python 同學"
ROOM   = sys.argv[2] if len(sys.argv) > 2 else "demo"
PREFIX = f"pyl-academy/v1/{ROOM}/"
BROKER = "broker.emqx.io"      # 公開測試伺服器(請勿傳個資)
PORT   = 1883                  # Python 直接用 TCP;網頁才需要 WebSocket(8084)

In [ ]:
MY_ID  = "py-" + uuid.uuid4().hex[:8]

### 連上時:訂閱與上線
這是回呼函式 callback:先寫好、登記給套件,等事情發生時由套件來呼叫你。連上 broker 時,訂閱聊天主題,再發布一則保留訊息宣告上線,網頁的在線名單就會出現你。寫在連上時才做的好處是:斷線重連後,它會自動再訂閱一次。

### 收到訊息:印出別人的話
broker 推來的內容是 bytes,先轉文字再用 json 解開,並取出名字和內容。這些包在 try 裡,格式不對就跳過,不然一則亂送的訊息就會讓背景收訊停掉。自己發的訊息也會被送回來,所以 id 等於自己的編號就不印。印的時候用反斜線 r 蓋掉輸入提示再補回,和原本 ChatClient 的技巧一樣。

In [ ]:
# ====== 2. 連上 broker 時:訂閱聊天主題、宣告上線 ======
def 連上了(client, userdata, flags, reason_code, properties):
    print(f"已連上 {BROKER},房間「{ROOM}」。輸入訊息按 Enter 送出,輸入 /quit 離開。")
    client.subscribe(PREFIX + "chat")
    client.publish(PREFIX + "online/" + MY_ID, NAME, retain=True)


# ====== 3. 收到訊息:broker 會把同主題的訊息推過來 ======
def 收到訊息(client, userdata, msg):
    try:
        m = json.loads(msg.payload.decode("utf-8"))
        name, text = str(m["name"]), str(m["text"])
    except (ValueError, KeyError, TypeError):
        return                      # 公開 broker 誰都能亂送:格式不對就跳過,不要讓收訊停掉
    if m.get("id") != MY_ID:        # 用 ID 判斷是不是自己,同名的人也分得出來
        print(f"\r{name}:{text}\n> ", end="", flush=True)

### 登記回呼與遺囑
建立 Client 時指定第二版的回呼格式,再把剛剛寫好的兩個函式登記上去。接著設定遺囑:這支程式當掉或斷網時,broker 會替它發布空字串,網頁的在線名單就會自動把它移除。遺囑一定要在連線之前設定。

In [ ]:
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=MY_ID)
client.on_connect = 連上了
client.on_message = 收到訊息
# 遺囑:這支程式當掉或斷網時,broker 會幫我發布「空字串」→ 網頁的在線名單自動把我移除
client.will_set(PREFIX + "online/" + MY_ID, "", retain=True)

### 連線與背景收訊
keepalive 三十秒的意思是:每三十秒至少跟 broker 打一次招呼,太久沒聲音就會被判定斷線,遺囑也就發出去了。loop_start 開一條背景執行緒收訊息;原本 ChatClient 要自己寫 threading,這裡一行搞定。

In [ ]:
client.connect(BROKER, PORT, keepalive=30)

In [ ]:
client.loop_start()            # 背景執行緒負責收訊息;主執行緒負責 input(),不用自己開 thread

### 主迴圈:讀鍵盤、發布
主執行緒只負責讀鍵盤。輸入斜線 quit 就離開;有打字就連同我的編號打包成 JSON 發布。ensure_ascii 設成 False 讓中文直接送,時間乘一千變成毫秒,和網頁的 Date.now 單位一致。收訊在背景執行緒,所以等你打字時,別人的訊息照樣會跳出來。

### 收尾:主動下線
不管怎麼離開,finally 都會執行。先發布空字串的保留訊息把自己從名單清掉,等零點三秒讓這則訊息送出去,再停掉背景執行緒、斷線。要自己清的原因是:用 disconnect 正常說再見時,broker 不會發遺囑。

In [ ]:
# ====== 4. 主迴圈:讀鍵盤、發布到聊天主題 ======
try:
    while True:
        text = input("> ").strip()
        if text == "/quit":
            break
        if text:
            client.publish(PREFIX + "chat", json.dumps({"id": MY_ID, "name": NAME, "text": text, "ts": int(time.time() * 1000)}, ensure_ascii=False))
finally:
    client.publish(PREFIX + "online/" + MY_ID, "", retain=True)   # 主動下線
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()

## 小結
Python 版和網頁版用同一套主題,所以能互通;paho-mqtt 的回呼加 loop_start,取代了原本自己寫的收訊執行緒。下一節進到多人遊戲,看 broker 怎麼取代 pygame_multiplayer 伺服器的連線管理。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/1) 看逐句拆解、做練習。